# D245 — MovieLens Hive Exercise

In this exercise you will ingest MovieLens `movies.csv` and `ratings.csv`, register external Hive tables, identify highly rated movies, and materialize the final result as a managed Parquet table using CTAS.

This is a learner exercise. It provides requirements, checkpoints, and hints—but not the completed HiveQL solution.

## Learning objectives

By the end, you should be able to:

- prepare Windows-hosted CSV files for Hive from WSL;
- remove CSV headers with `sed`;
- upload files to dedicated HDFS directories;
- create external Hive tables over CSV data;
- aggregate ratings with `AVG`, `COUNT`, `GROUP BY`, and `HAVING`;
- use Hive's session-scoped temporary-table equivalent to a temporary view;
- join the aggregated result with movie metadata;
- store a curated result using `CREATE TABLE AS SELECT` (CTAS).

## Final requirement

Create a managed Hive table containing these columns, in this order:

| Column | Meaning |
|---|---|
| `movieId` | Movie identifier |
| `title` | Movie title |
| `avg_rating` | Average rating for the movie |
| `total_ratings` | Number of distinct users who rated it |

A movie qualifies only when:

- it was rated by **at least 100 distinct users**; and
- its average rating is **at least 4.0**.

Store the final table as Parquet and order displayed results by highest average rating, then highest rating count.

## 0. Source-data facts

The files were inspected at these Windows paths:

```text
C:\data\movielens\ml-latest-small\movies\movies.csv
C:\data\movielens\ml-latest-small\ratings\ratings.csv
```

From WSL, the same files are available under `/mnt/c`:

```text
/mnt/c/data/movielens/ml-latest-small/movies/movies.csv
/mnt/c/data/movielens/ml-latest-small/ratings/ratings.csv
```

The source files contain 9,743 and 100,837 lines respectively, including one header line in each file.

### Source schemas

`movies.csv`:

```text
movieId,title,genres
```

`ratings.csv`:

```text
userId,movieId,rating,timestamp
```

The Unix timestamp is not required in the final output, but your external ratings table should still represent it with a suitable numeric type.

## 1. Service spot-check

In [ ]:
%%bash
jps
ss -lnt | grep -E ':(9083|10000)\b' || true
hdfs dfsadmin -report | grep -E 'Live datanodes|Name:'
yarn node -list

Expected: one live DataNode, one running YARN node, the Hive metastore, and HiveServer2 listening on port `10000`.

## 2. Inspect the source files

Run these commands before modifying or copying anything.

In [ ]:
%%bash
MOVIES_SOURCE=/mnt/c/data/movielens/ml-latest-small/movies/movies.csv
RATINGS_SOURCE=/mnt/c/data/movielens/ml-latest-small/ratings/ratings.csv
ls -lh "$MOVIES_SOURCE" "$RATINGS_SOURCE"
echo '--- movies sample ---'
sed -n '1,5p' "$MOVIES_SOURCE"
echo '--- ratings sample ---'
sed -n '1,5p' "$RATINGS_SOURCE"
echo '--- line counts including headers ---'
wc -l "$MOVIES_SOURCE" "$RATINGS_SOURCE"

Notice that some movie titles are quoted and may contain commas. Therefore, `ROW FORMAT DELIMITED FIELDS TERMINATED BY ','` is not sufficient for `movies.csv`; use a CSV-aware SerDe.

## 3. Remove the first line with `sed`

`sed '1d'` deletes line 1 from the command's output. The original Windows files remain unchanged; the headerless copies are written under WSL's `/tmp` directory.

In [ ]:
%%bash
MOVIES_SOURCE=/mnt/c/data/movielens/ml-latest-small/movies/movies.csv
RATINGS_SOURCE=/mnt/c/data/movielens/ml-latest-small/ratings/ratings.csv
sed '1d' "$MOVIES_SOURCE" > /tmp/d245_movies.csv
sed '1d' "$RATINGS_SOURCE" > /tmp/d245_ratings.csv
echo '--- headerless samples ---'
sed -n '1,3p' /tmp/d245_movies.csv
sed -n '1,3p' /tmp/d245_ratings.csv
echo '--- data-row counts ---'
wc -l /tmp/d245_movies.csv /tmp/d245_ratings.csv

Checkpoint: the temporary files should contain exactly:

- `d245_movies.csv`: **9,742** rows;
- `d245_ratings.csv`: **100,836** rows.

## 4. Upload the headerless files to HDFS

Use separate directories so each external table reads only its own file. The cleanup is limited to the exact D245 paths and makes the staging process repeatable.

In [ ]:
%%bash
MOVIES_HDFS=/user/hive/external/d245_movielens/movies
RATINGS_HDFS=/user/hive/external/d245_movielens/ratings
hdfs dfs -rm -r -f /user/hive/external/d245_movielens
hdfs dfs -mkdir -p "$MOVIES_HDFS" "$RATINGS_HDFS"
hdfs dfs -put /tmp/d245_movies.csv "$MOVIES_HDFS"/
hdfs dfs -put /tmp/d245_ratings.csv "$RATINGS_HDFS"/
hdfs dfs -ls -R /user/hive/external/d245_movielens

Verify HDFS row counts before creating tables.

In [ ]:
%%bash
hdfs dfs -cat /user/hive/external/d245_movielens/movies/d245_movies.csv | wc -l
hdfs dfs -cat /user/hive/external/d245_movielens/ratings/d245_ratings.csv | wc -l

## 5. Create the exercise database

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/default' -n "$USER" --silent=true -e "
CREATE DATABASE IF NOT EXISTS movielens_exercise
COMMENT 'D245 MovieLens learner exercise';
DESCRIBE DATABASE EXTENDED movielens_exercise;
"

# Exercise A — Create the external tables

Create these two external tables in `movielens_exercise`:

1. `movies_external`, pointing at `/user/hive/external/d245_movielens/movies`;
2. `ratings_external`, pointing at `/user/hive/external/d245_movielens/ratings`.

Requirements:

- represent every source column;
- choose reasonable Hive types;
- use `EXTERNAL`;
- do not configure header skipping—the headers were already removed;
- use a CSV-aware SerDe for movies because quoted titles can contain commas;
- use the correct HDFS `LOCATION` for each table.

In [ ]:
%%bash
# TODO: Write and run your CREATE EXTERNAL TABLE statements.
# You can place a multi-statement HiveQL script inside:
# beeline -u 'jdbc:hive2://localhost:10000/movielens_exercise' -n "$USER" -e " ... "

### Hints for Exercise A

- For movies, investigate `org.apache.hadoop.hive.serde2.OpenCSVSerde`.
- `OpenCSVSerde` commonly exposes CSV fields as strings. If you choose string identifiers, cast `movieId` when joining to a numeric identifier.
- Ratings do not contain quoted comma-bearing text, so a standard comma-delimited row format is sufficient.
- A rating such as `4.5` needs a fractional numeric type.
- The Unix timestamp values require a type larger than a 32-bit `INT`.
- `DESCRIBE FORMATTED table_name` should show `EXTERNAL_TABLE` and the intended location.

### Validation checkpoint A

Write queries that prove:

- `movies_external` contains **9,742** rows;
- `ratings_external` contains **100,836** rows;
- movie titles containing commas remain in a single `title` column;
- the minimum and maximum rating values are sensible.

In [ ]:
%%bash
# TODO: Add COUNT, sample-title, MIN(rating), and MAX(rating) validation queries.

# Exercise B — Build the qualifying-ratings temporary result

Aggregate `ratings_external` by movie and retain only movies meeting both conditions:

- at least 100 distinct users;
- average rating at least 4.0.

The temporary result needs three columns: the movie identifier, `avg_rating`, and `total_ratings`.

### Important: temporary view vs Hive temporary table

Apache Hive does not support Spark SQL's `CREATE TEMP VIEW`. Use a Hive **session-scoped temporary table** as the equivalent for this exercise. Create it with CTAS from your aggregation query.

A temporary table exists only in its current Beeline session. Therefore, create the temporary table and the final CTAS table within the **same Beeline connection/script**. If Beeline exits between those statements, the temporary table disappears.

### Hints for Exercise B

- Required operations: `GROUP BY`, `AVG`, `COUNT(DISTINCT ...)`, and `HAVING`.
- `WHERE` filters rows before aggregation; `HAVING` filters aggregated groups.
- Give aggregate expressions the exact aliases `avg_rating` and `total_ratings`.
- Use `CREATE TEMPORARY TABLE ... STORED AS ORC AS SELECT ...`.
- For an alternative that lasts only one statement, a CTE beginning with `WITH` can act as a query-scoped temporary result—but it cannot be reused by a later statement.

### Learner template—not a solution

Complete all placeholders and keep the final CTAS statement in the same script as the temporary-table statement.

```sql
USE movielens_exercise;

CREATE TEMPORARY TABLE qualifying_ratings_temp
STORED AS ORC
AS
SELECT
    /* movie identifier */,
    /* average expression and alias */,
    /* distinct-user count and alias */
FROM ratings_external
/* grouping */
/* aggregate filters */;

-- Do not exit this Beeline session yet.
```

# Exercise C — Join with movies and create the final table

Join `qualifying_ratings_temp` with `movies_external`. Use CTAS to create a managed table named `best_rated_movies` stored as Parquet.

The table must contain exactly:

```text
movieId, title, avg_rating, total_ratings
```

Do not include `genres` in the final table.

### Hints for Exercise C

- Drop `best_rated_movies` before CTAS so the exercise is repeatable.
- Join the two datasets using their movie identifiers.
- If `movies_external.movieId` is a string and the ratings identifier is numeric, cast one side to a compatible type.
- Qualify ambiguous column names with table aliases.
- `CREATE TABLE ... STORED AS PARQUET AS SELECT ...` both creates and populates the table.
- `ORDER BY` controls query output, not a permanent physical row order in a distributed table. Apply ordering when displaying the final results.

### Extend your same-session script

```sql
DROP TABLE IF EXISTS best_rated_movies;

CREATE TABLE best_rated_movies
STORED AS PARQUET
AS
SELECT
    /* four required output columns */
FROM /* movies table and alias */
JOIN /* temporary table and alias */
ON /* compatible movie identifiers */;

SELECT *
FROM best_rated_movies
ORDER BY avg_rating DESC, total_ratings DESC;
```

In [ ]:
%%bash
# TODO: Submit your completed Exercise B + Exercise C HiveQL in ONE Beeline call.
# The temporary table must be created and consumed in the same session.

# Exercise D — Validate the final table

Write queries or catalog commands to verify all of the following:

1. The table exists in `movielens_exercise`.
2. It is a managed table.
3. Its storage format is Parquet.
4. It has exactly four columns with the requested names.
5. Every `avg_rating` is at least 4.0.
6. Every `total_ratings` is at least 100.
7. There are no null movie titles.
8. Displayed results are sorted by `avg_rating` descending and `total_ratings` descending.

In [ ]:
%%bash
# TODO: Add DESCRIBE FORMATTED and data-quality validation queries.

## Debugging hints

**Movie titles are split at commas**  
Your movies table probably uses a simple comma delimiter. Use a CSV-aware SerDe and recreate the table.

**The join returns no rows**  
Inspect `DESCRIBE` output and sample movie identifiers from both sides. Check whether one is `STRING` and the other is numeric.

**Temporary table not found**  
It was probably created in a previous Beeline session. Submit temporary-table creation and final CTAS in one Beeline call or interactive session.

**Too many movies qualify**  
Check that both aggregate conditions are in `HAVING` and that the user count is distinct.

**CTAS says the table already exists**  
Drop only `best_rated_movies`, then rerun the same-session script.

**Header text appears as a row**  
Confirm that the external tables point to the headerless `/tmp` copies uploaded under the D245 HDFS directory.

## Optional extension challenges

After finishing the required work, try one or more of these without changing the original final table:

- include genres and find the most common genre among qualifying movies;
- compare `COUNT(*)` with `COUNT(DISTINCT userId)` and explain the result;
- change the minimum user threshold to 50 and compare the number of qualifying movies;
- use `EXPLAIN` on the aggregation and final join;
- create a second CTAS table partitioned or bucketed according to a design you can justify.

## Cleanup guidance

Keep the result for review unless instructed to clean up. Your managed `best_rated_movies` data is removed when that table is dropped. Dropping the two external tables does not remove their CSV files.

The staged source files remain under `/user/hive/external/d245_movielens`. Remove that exact directory only after the exercise no longer needs it.